# Food Desert Case Study

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/07-food-desert-case-study.ipynb)

This notebook applies SocialMapper to a real-world equity analysis: identifying food deserts and analyzing their demographic characteristics.

## What is a Food Desert?

A **food desert** is an area with limited access to affordable, nutritious food. The USDA defines food deserts as:
- **Urban**: More than 1 mile from a supermarket
- **Rural**: More than 10 miles from a supermarket

We'll use SocialMapper to analyze food access patterns and understand who is affected.

## Setup

In [ ]:
!pip install -q socialmapper[routing] folium

In [ ]:
import os
import json
os.environ["SOCIALMAPPER_DEMO_MODE"] = "true"

from socialmapper import (
    create_isochrone,
    get_poi,
    get_census_blocks,
    get_census_data,
    create_map
)
from shapely.geometry import shape, Point
from IPython.display import Image, display
import folium

print("Ready for food desert analysis!")

## Part 1: Single Location Analysis

Let's start by analyzing food access for a specific location.

In [ ]:
# Define study location
location = "Detroit, MI"

print(f"Food Access Analysis: {location}")
print("=" * 50)

### Step 1: Define Walking Distance

We'll use a 15-minute walk (~1 km) as our accessibility threshold.

In [ ]:
# Create walking isochrone (15 minutes ≈ 1 mile)
walk_area = create_isochrone(
    location=location,
    travel_time=15,
    travel_mode="walk"
)

print(f"Walking area (15 min): {walk_area['properties']['area_sq_km']:.2f} km²")

### Step 2: Find Food Sources

In [ ]:
# Find grocery stores and supermarkets
food_sources = get_poi(
    location=location,
    categories=["shopping"],
    travel_time=15,
    limit=50
)

print(f"Food sources within 15-min walk: {len(food_sources)}")

if food_sources:
    print("\nNearest stores:")
    for store in sorted(food_sources, key=lambda x: x['distance_km'])[:5]:
        print(f"  - {store['name']}: {store['distance_km']:.2f} km")
else:
    print("\nWARNING: No grocery stores within walking distance!")
    print("This area may be a food desert.")

### Step 3: Analyze Demographics

In [ ]:
# Get census blocks and demographics
blocks = get_census_blocks(polygon=walk_area)
geoids = [b['geoid'] for b in blocks]

census_result = get_census_data(
    location=geoids,
    variables=["population", "median_income", "total_households"]
)

# Combine data
for block in blocks:
    data = census_result.data.get(block['geoid'], {})
    block['population'] = data.get('population', 0) or 0
    block['median_income'] = data.get('median_income', 0) or 0
    block['households'] = data.get('total_households', 0) or 0

# Calculate totals
total_pop = sum(b['population'] for b in blocks)
total_households = sum(b['households'] for b in blocks)
incomes = [b['median_income'] for b in blocks if b['median_income'] > 0]

print(f"\nDemographic Profile:")
print(f"  Population affected: {total_pop:,}")
print(f"  Households: {total_households:,}")
if incomes:
    print(f"  Average median income: ${sum(incomes)//len(incomes):,}")

### Step 4: Food Desert Classification

In [ ]:
def classify_food_access(num_stores, income, is_urban=True):
    """
    Classify food access based on USDA criteria.
    
    Food deserts are low-income areas with limited access.
    """
    # Low income threshold (median income below $50,000)
    low_income = income < 50000
    
    # Low access (fewer than 2 stores within walking distance)
    low_access = num_stores < 2
    
    if low_income and low_access:
        return "FOOD DESERT", "Low income AND low access"
    elif low_access:
        return "LOW ACCESS", "Limited food access"
    elif low_income:
        return "LOW INCOME", "Limited income but adequate access"
    else:
        return "ADEQUATE", "Good income and food access"

# Classify this area
avg_income = sum(incomes) // len(incomes) if incomes else 0
classification, reason = classify_food_access(len(food_sources), avg_income)

print(f"\n{'='*50}")
print(f"FOOD ACCESS CLASSIFICATION: {classification}")
print(f"{'='*50}")
print(f"Reason: {reason}")
print(f"  - Grocery stores: {len(food_sources)}")
print(f"  - Avg median income: ${avg_income:,}")
print(f"  - Population affected: {total_pop:,}")

## Part 2: Multi-Area Comparison

Compare food access across different neighborhoods.

In [ ]:
def analyze_food_access(location):
    """Analyze food access for a location."""
    
    # Get walkable area
    walk_area = create_isochrone(
        location=location,
        travel_time=15,
        travel_mode="walk"
    )
    
    # Find food sources
    food_sources = get_poi(
        location=location,
        categories=["shopping"],
        travel_time=15,
        limit=50
    )
    
    # Get demographics
    blocks = get_census_blocks(polygon=walk_area)
    geoids = [b['geoid'] for b in blocks]
    census = get_census_data(geoids, variables=["population", "median_income"])
    
    # Calculate metrics
    total_pop = sum(
        census.data.get(g, {}).get('population', 0) or 0
        for g in geoids
    )
    incomes = [
        census.data.get(g, {}).get('median_income', 0)
        for g in geoids
        if census.data.get(g, {}).get('median_income', 0)
    ]
    avg_income = sum(incomes) // len(incomes) if incomes else 0
    
    # Classify
    classification, _ = classify_food_access(len(food_sources), avg_income)
    
    return {
        "location": location,
        "food_stores": len(food_sources),
        "population": total_pop,
        "avg_income": avg_income,
        "classification": classification,
        "area_sq_km": walk_area['properties']['area_sq_km']
    }

In [ ]:
# Compare multiple neighborhoods
neighborhoods = [
    "Downtown Detroit, MI",
    "Midtown Detroit, MI",
    "Corktown, Detroit, MI",
    "Hamtramck, MI"
]

print("Neighborhood Food Access Comparison")
print("=" * 70)

results = []
for hood in neighborhoods:
    result = analyze_food_access(hood)
    results.append(result)
    
    print(f"\n{result['location']}:")
    print(f"  Food stores: {result['food_stores']}")
    print(f"  Population: {result['population']:,}")
    print(f"  Avg Income: ${result['avg_income']:,}")
    print(f"  Status: {result['classification']}")

In [ ]:
# Summary statistics
food_deserts = [r for r in results if r['classification'] == "FOOD DESERT"]
low_access = [r for r in results if r['classification'] in ["FOOD DESERT", "LOW ACCESS"]]

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"Total areas analyzed: {len(results)}")
print(f"Food deserts: {len(food_deserts)}")
print(f"Low access areas: {len(low_access)}")

# Population in food deserts
pop_in_deserts = sum(r['population'] for r in food_deserts)
total_pop = sum(r['population'] for r in results)

if total_pop > 0:
    print(f"\nPopulation in food deserts: {pop_in_deserts:,} ({pop_in_deserts/total_pop*100:.1f}%)")
    print(f"Total population analyzed: {total_pop:,}")

## Part 3: Visualization

In [ ]:
# Create a comprehensive food access map
location = "Detroit, MI"

# Get larger study area
study_area = create_isochrone(location, travel_time=30, travel_mode="drive")
blocks = get_census_blocks(polygon=study_area)

# Get census data
geoids = [b['geoid'] for b in blocks]
census = get_census_data(geoids, variables=["population", "median_income"])

# Add data to blocks
for block in blocks:
    data = census.data.get(block['geoid'], {})
    block['population'] = data.get('population', 0) or 0
    block['median_income'] = data.get('median_income', 0) or 0

# Create income map
valid_blocks = [b for b in blocks if b['median_income'] > 0]

income_map = create_map(
    data=valid_blocks,
    column="median_income",
    title="Median Income - Detroit Metro Area",
    save_path="detroit_income.png"
)

display(Image(filename="detroit_income.png"))

### Interactive Food Access Map

In [ ]:
# Create interactive map with grocery stores
center = (42.3314, -83.0458)  # Detroit
m = folium.Map(location=center, zoom_start=12)

# Get grocery stores for larger area
all_groceries = get_poi(
    location=center,
    categories=["shopping"],
    limit=100
)

# Add grocery store markers
for store in all_groceries:
    folium.Marker(
        location=[store['lat'], store['lon']],
        popup=f"{store['name']}<br>Distance: {store['distance_km']:.2f} km",
        icon=folium.Icon(color='green', icon='shopping-cart', prefix='fa')
    ).add_to(m)

# Add census blocks colored by income
for block in valid_blocks[:50]:  # Limit for performance
    income = block['median_income']
    
    # Color by income level
    if income < 30000:
        color = '#d73027'  # Red - low income
    elif income < 50000:
        color = '#fc8d59'  # Orange
    elif income < 75000:
        color = '#fee08b'  # Yellow
    else:
        color = '#91cf60'  # Green - high income
    
    folium.GeoJson(
        block['geometry'],
        style_function=lambda x, c=color: {
            'fillColor': c,
            'color': 'gray',
            'weight': 0.5,
            'fillOpacity': 0.5
        },
        tooltip=f"Income: ${income:,}<br>Pop: {block['population']:,}"
    ).add_to(m)

# Add legend
legend_html = '''
<div style="position: fixed; bottom: 50px; left: 50px; z-index: 1000;
            background-color: white; padding: 10px; border-radius: 5px;
            border: 2px solid gray;">
    <b>Legend</b><br>
    <i style="color: green">●</i> Grocery Store<br>
    <div style="background: #d73027; width: 20px; height: 10px; display: inline-block;"></div> Income < $30k<br>
    <div style="background: #fc8d59; width: 20px; height: 10px; display: inline-block;"></div> Income $30-50k<br>
    <div style="background: #fee08b; width: 20px; height: 10px; display: inline-block;"></div> Income $50-75k<br>
    <div style="background: #91cf60; width: 20px; height: 10px; display: inline-block;"></div> Income > $75k
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

print(f"Map shows {len(all_groceries)} grocery stores")
m

## Part 4: Food Desert Score

In [ ]:
def calculate_food_desert_score(location):
    """
    Calculate a food desert score (0-100) for a location.
    
    Higher scores indicate worse food access.
    
    Components:
    - Store access (0-40 points)
    - Income level (0-30 points)
    - Store density (0-30 points)
    """
    
    # Get data
    walk_area = create_isochrone(location, travel_time=15, travel_mode="walk")
    area_km = walk_area['properties']['area_sq_km']
    
    food_sources = get_poi(
        location=location,
        categories=["shopping"],
        travel_time=15,
        limit=50
    )
    
    blocks = get_census_blocks(polygon=walk_area)
    geoids = [b['geoid'] for b in blocks]
    census = get_census_data(geoids, variables=["population", "median_income"])
    
    # Calculate metrics
    total_pop = sum(
        census.data.get(g, {}).get('population', 0) or 0
        for g in geoids
    )
    incomes = [
        census.data.get(g, {}).get('median_income', 0)
        for g in geoids
        if census.data.get(g, {}).get('median_income', 0)
    ]
    avg_income = sum(incomes) // len(incomes) if incomes else 0
    
    # Score components
    score = 0
    details = []
    
    # Store access (0-40 points, more stores = lower score)
    if len(food_sources) == 0:
        store_score = 40
    elif len(food_sources) == 1:
        store_score = 30
    elif len(food_sources) <= 3:
        store_score = 15
    else:
        store_score = 0
    score += store_score
    details.append(f"Store access: {store_score}/40")
    
    # Income level (0-30 points, lower income = higher score)
    if avg_income < 25000:
        income_score = 30
    elif avg_income < 40000:
        income_score = 20
    elif avg_income < 60000:
        income_score = 10
    else:
        income_score = 0
    score += income_score
    details.append(f"Income level: {income_score}/30")
    
    # Store density (0-30 points, fewer stores per km² = higher score)
    density = len(food_sources) / area_km if area_km > 0 else 0
    if density < 0.5:
        density_score = 30
    elif density < 1.0:
        density_score = 20
    elif density < 2.0:
        density_score = 10
    else:
        density_score = 0
    score += density_score
    details.append(f"Store density: {density_score}/30")
    
    # Classify
    if score >= 70:
        risk = "CRITICAL"
    elif score >= 50:
        risk = "HIGH"
    elif score >= 30:
        risk = "MODERATE"
    else:
        risk = "LOW"
    
    return {
        "location": location,
        "score": score,
        "risk": risk,
        "details": details,
        "food_stores": len(food_sources),
        "population": total_pop,
        "avg_income": avg_income,
        "area_km": area_km
    }

In [ ]:
# Calculate scores for multiple areas
test_locations = [
    "Detroit, MI",
    "Manhattan, NY",
    "Beverly Hills, CA",
    "South Side Chicago, IL"
]

print("Food Desert Risk Assessment")
print("=" * 70)

all_scores = []
for loc in test_locations:
    result = calculate_food_desert_score(loc)
    all_scores.append(result)
    
    print(f"\n{result['location']}")
    print(f"  Score: {result['score']}/100 - {result['risk']} RISK")
    for detail in result['details']:
        print(f"    {detail}")
    print(f"  Food stores: {result['food_stores']}, Population: {result['population']:,}")

## Part 5: Recommendations

In [ ]:
def generate_recommendations(score_result):
    """Generate recommendations based on food desert analysis."""
    
    recommendations = []
    
    if score_result['food_stores'] == 0:
        recommendations.append("URGENT: No grocery stores within walking distance")
        recommendations.append("  - Consider mobile grocery programs")
        recommendations.append("  - Evaluate locations for new grocery development")
        recommendations.append("  - Implement community garden initiatives")
    
    elif score_result['food_stores'] < 3:
        recommendations.append("Limited food access detected")
        recommendations.append("  - Support existing grocery stores with incentives")
        recommendations.append("  - Explore farmers market opportunities")
    
    if score_result['avg_income'] < 40000:
        recommendations.append("Low-income area identified")
        recommendations.append("  - Expand SNAP retailer participation")
        recommendations.append("  - Consider food assistance programs")
        recommendations.append("  - Partner with food banks for distribution")
    
    if score_result['score'] >= 50:
        recommendations.append("High-priority area for intervention")
        recommendations.append("  - Conduct detailed community needs assessment")
        recommendations.append("  - Engage local stakeholders for solutions")
    
    if not recommendations:
        recommendations.append("Area has adequate food access")
        recommendations.append("  - Monitor for changes in store availability")
        recommendations.append("  - Maintain support for healthy food initiatives")
    
    return recommendations

# Generate recommendations for each area
print("\n" + "=" * 70)
print("RECOMMENDATIONS")
print("=" * 70)

for result in all_scores:
    print(f"\n{result['location']} ({result['risk']} Risk):")
    recs = generate_recommendations(result)
    for rec in recs:
        print(f"  {rec}")

## Part 6: Export Report

In [ ]:
# Create comprehensive report
report = {
    "title": "Food Desert Analysis Report",
    "date": "2025-01-24",
    "summary": {
        "areas_analyzed": len(all_scores),
        "high_risk_areas": len([s for s in all_scores if s['risk'] in ['HIGH', 'CRITICAL']]),
        "total_population_at_risk": sum(
            s['population'] for s in all_scores 
            if s['risk'] in ['HIGH', 'CRITICAL']
        )
    },
    "areas": [
        {
            "location": s['location'],
            "score": s['score'],
            "risk_level": s['risk'],
            "food_stores": s['food_stores'],
            "population": s['population'],
            "avg_income": s['avg_income'],
            "recommendations": generate_recommendations(s)
        }
        for s in all_scores
    ]
}

# Save report
with open("food_desert_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("Report saved: food_desert_report.json")
print("\nReport Summary:")
print(f"  Areas analyzed: {report['summary']['areas_analyzed']}")
print(f"  High-risk areas: {report['summary']['high_risk_areas']}")
print(f"  Population at risk: {report['summary']['total_population_at_risk']:,}")

## Conclusion

This case study demonstrated how to use SocialMapper for equity analysis:

1. **Isochrones** define walkable access areas
2. **POI queries** find food sources
3. **Census data** provides demographic context
4. **Scoring systems** quantify food desert risk
5. **Visualization** communicates findings

### Key Insights

- Food deserts disproportionately affect low-income communities
- Walking distance is critical for car-free households
- Solutions require understanding both access AND affordability

### Next Steps

- Expand analysis to entire cities
- Include convenience stores and farmers markets
- Track changes over time
- Integrate with policy recommendations